# BRAID Library Testing Notebook

This notebook provides a testing environment for the BRAID library, allowing visualization of 2D cross-sections from Arrow shard files stored in GCS.

## Setup

Make sure you have installed the braid library with notebook dependencies:
```bash
cd ../braid
pip install -e .[notebook,gcs]
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import sys
import os
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
import gcsfs

# Add braid to path
braid_path = Path('../braid/src').resolve()
if str(braid_path) not in sys.path:
    sys.path.insert(0, str(braid_path))

from braid import ShardReader, LabelType
from braid.exceptions import ChunkNotFoundError, DecompressionError

## Configuration

Set up your GCS paths and dataset parameters.

In [ ]:
# GCS Configuration
GCS_BUCKET = "your-bucket-name"  # Replace with your GCS bucket
SHARD_PREFIX = "path/to/shard/files/"  # Replace with your shard file prefix

# Dataset Parameters
CHUNK_SHAPE = (64, 64, 64)  # Standard DVID chunk shape
SHARD_SHAPE = (2048, 2048, 2048)  # Standard shard shape

# Example shard coordinates (adjust for your dataset)
SHARD_X, SHARD_Y, SHARD_Z = 0, 0, 0  # Which shard to examine

print(f"Configuration:")
print(f"  GCS Bucket: {GCS_BUCKET}")
print(f"  Shard Prefix: {SHARD_PREFIX}")
print(f"  Chunk Shape: {CHUNK_SHAPE}")
print(f"  Shard Shape: {SHARD_SHAPE}")
print(f"  Target Shard: ({SHARD_X}, {SHARD_Y}, {SHARD_Z})")

## Initialize BRAID Reader

Load a specific shard for testing.

In [ ]:
# Construct GCS paths for arrow and csv files
shard_name = f"shard_{SHARD_X}_{SHARD_Y}_{SHARD_Z}"
arrow_path = f"gs://{GCS_BUCKET}/{SHARD_PREFIX}{shard_name}.arrow"
csv_path = f"gs://{GCS_BUCKET}/{SHARD_PREFIX}{shard_name}.csv"

print(f"Loading shard from:")
print(f"  Arrow: {arrow_path}")
print(f"  CSV: {csv_path}")

try:
    # Initialize reader with GCS paths
    reader = ShardReader(arrow_path, csv_path)
    
    print(f"\nShard loaded successfully!")
    print(f"  Total chunks: {reader.chunk_count}")
    print(f"  Available chunks (first 10): {reader.available_chunks[:10]}")
    
    # Get shard bounds
    if reader.available_chunks:
        chunks = np.array(reader.available_chunks)
        min_coords = chunks.min(axis=0) * CHUNK_SHAPE[0]
        max_coords = (chunks.max(axis=0) + 1) * CHUNK_SHAPE[0]
        print(f"  Shard bounds: {min_coords} to {max_coords}")
        
except Exception as e:
    print(f"Error loading shard: {e}")
    reader = None

## Visualization Functions

Helper functions for creating 2D cross-sections and pseudo-coloring.

In [ ]:
def create_pseudocolor_map(labels):
    """Create a colormap for pseudo-coloring unique labels."""
    unique_labels = np.unique(labels)
    n_labels = len(unique_labels)
    
    # Create a colormap with distinct colors
    colors = plt.cm.Set3(np.linspace(0, 1, max(12, n_labels)))
    # Set background (label 0) to black
    if 0 in unique_labels:
        colors[0] = [0, 0, 0, 1]  # Black for background
    
    return ListedColormap(colors[:n_labels]), unique_labels

def extract_2d_slice(reader, top_left, bottom_right, axis='z', slice_pos=None, label_type=LabelType.LABELS):
    """Extract a 2D slice from the shard data.
    
    Args:
        reader: ShardReader instance
        top_left: (x, y, z) coordinates of top-left corner
        bottom_right: (x, y, z) coordinates of bottom-right corner
        axis: Which axis to slice along ('x', 'y', or 'z')
        slice_pos: Position along the slice axis (if None, uses middle)
        label_type: LabelType.LABELS or LabelType.SUPERVOXELS
    
    Returns:
        2D numpy array of the slice
    """
    x1, y1, z1 = top_left
    x2, y2, z2 = bottom_right
    
    # Determine slice position
    if slice_pos is None:
        if axis == 'x':
            slice_pos = (x1 + x2) // 2
        elif axis == 'y':
            slice_pos = (y1 + y2) // 2
        else:  # axis == 'z'
            slice_pos = (z1 + z2) // 2
    
    # Create output array
    if axis == 'x':
        slice_shape = (z2 - z1, y2 - y1)
        coords_iter = [(slice_pos, y, z) for z in range(z1, z2) for y in range(y1, y2)]
    elif axis == 'y':
        slice_shape = (z2 - z1, x2 - x1)
        coords_iter = [(x, slice_pos, z) for z in range(z1, z2) for x in range(x1, x2)]
    else:  # axis == 'z'
        slice_shape = (y2 - y1, x2 - x1)
        coords_iter = [(x, y, slice_pos) for y in range(y1, y2) for x in range(x1, x2)]
    
    slice_data = np.zeros(slice_shape, dtype=np.uint64)
    
    # Process chunks that intersect with our slice
    chunk_size = CHUNK_SHAPE[0]  # Assuming cubic chunks
    
    for chunk_x, chunk_y, chunk_z in reader.available_chunks:
        # Check if chunk intersects with our slice region
        chunk_bounds = (
            chunk_x * chunk_size, (chunk_x + 1) * chunk_size,
            chunk_y * chunk_size, (chunk_y + 1) * chunk_size,
            chunk_z * chunk_size, (chunk_z + 1) * chunk_size
        )
        
        # Check intersection based on slice axis
        if axis == 'x' and not (chunk_bounds[0] <= slice_pos < chunk_bounds[1] and
                               chunk_bounds[2] < y2 and chunk_bounds[3] > y1 and
                               chunk_bounds[4] < z2 and chunk_bounds[5] > z1):
            continue
        elif axis == 'y' and not (chunk_bounds[2] <= slice_pos < chunk_bounds[3] and
                                 chunk_bounds[0] < x2 and chunk_bounds[1] > x1 and
                                 chunk_bounds[4] < z2 and chunk_bounds[5] > z1):
            continue
        elif axis == 'z' and not (chunk_bounds[4] <= slice_pos < chunk_bounds[5] and
                                 chunk_bounds[0] < x2 and chunk_bounds[1] > x1 and
                                 chunk_bounds[2] < y2 and chunk_bounds[3] > y1):
            continue
        
        try:
            # Read chunk data
            chunk_data = reader.read_chunk(chunk_x, chunk_y, chunk_z, label_type=label_type)
            
            # Extract relevant slice from chunk
            if axis == 'x':
                local_x = slice_pos - chunk_x * chunk_size
                if 0 <= local_x < chunk_size:
                    chunk_slice = chunk_data[local_x, :, :]
                    # Map to output coordinates
                    y_start = max(0, chunk_y * chunk_size - y1)
                    z_start = max(0, chunk_z * chunk_size - z1)
                    y_end = min(slice_shape[1], y_start + chunk_size)
                    z_end = min(slice_shape[0], z_start + chunk_size)
                    
                    chunk_y_start = max(0, y1 - chunk_y * chunk_size)
                    chunk_z_start = max(0, z1 - chunk_z * chunk_size)
                    
                    slice_data[z_start:z_end, y_start:y_end] = chunk_slice[
                        chunk_z_start:chunk_z_start + (z_end - z_start),
                        chunk_y_start:chunk_y_start + (y_end - y_start)
                    ]
            elif axis == 'y':
                local_y = slice_pos - chunk_y * chunk_size
                if 0 <= local_y < chunk_size:
                    chunk_slice = chunk_data[:, local_y, :]
                    # Similar mapping for y-axis slice
                    x_start = max(0, chunk_x * chunk_size - x1)
                    z_start = max(0, chunk_z * chunk_size - z1)
                    x_end = min(slice_shape[1], x_start + chunk_size)
                    z_end = min(slice_shape[0], z_start + chunk_size)
                    
                    chunk_x_start = max(0, x1 - chunk_x * chunk_size)
                    chunk_z_start = max(0, z1 - chunk_z * chunk_size)
                    
                    slice_data[z_start:z_end, x_start:x_end] = chunk_slice[
                        chunk_z_start:chunk_z_start + (z_end - z_start),
                        chunk_x_start:chunk_x_start + (x_end - x_start)
                    ]
            else:  # axis == 'z'
                local_z = slice_pos - chunk_z * chunk_size
                if 0 <= local_z < chunk_size:
                    chunk_slice = chunk_data[:, :, local_z]
                    # Map to output coordinates
                    x_start = max(0, chunk_x * chunk_size - x1)
                    y_start = max(0, chunk_y * chunk_size - y1)
                    x_end = min(slice_shape[1], x_start + chunk_size)
                    y_end = min(slice_shape[0], y_start + chunk_size)
                    
                    chunk_x_start = max(0, x1 - chunk_x * chunk_size)
                    chunk_y_start = max(0, y1 - chunk_y * chunk_size)
                    
                    slice_data[y_start:y_end, x_start:x_end] = chunk_slice[
                        chunk_y_start:chunk_y_start + (y_end - y_start),
                        chunk_x_start:chunk_x_start + (x_end - x_start)
                    ]
                    
        except (ChunkNotFoundError, DecompressionError) as e:
            print(f"Warning: Could not read chunk ({chunk_x}, {chunk_y}, {chunk_z}): {e}")
            continue
    
    return slice_data

## Interactive Visualization

Create an interactive widget for exploring 2D slices.

In [ ]:
if reader is not None:
    # Get available coordinate ranges
    chunks = np.array(reader.available_chunks)
    chunk_size = CHUNK_SHAPE[0]
    
    min_coords = chunks.min(axis=0) * chunk_size
    max_coords = (chunks.max(axis=0) + 1) * chunk_size
    
    # Create widgets
    x1_widget = widgets.IntSlider(value=min_coords[0], min=min_coords[0], max=max_coords[0]-1, description='X1:')
    y1_widget = widgets.IntSlider(value=min_coords[1], min=min_coords[1], max=max_coords[1]-1, description='Y1:')
    z1_widget = widgets.IntSlider(value=min_coords[2], min=min_coords[2], max=max_coords[2]-1, description='Z1:')
    
    x2_widget = widgets.IntSlider(value=min_coords[0]+128, min=min_coords[0], max=max_coords[0], description='X2:')
    y2_widget = widgets.IntSlider(value=min_coords[1]+128, min=min_coords[1], max=max_coords[1], description='Y2:')
    z2_widget = widgets.IntSlider(value=min_coords[2]+128, min=min_coords[2], max=max_coords[2], description='Z2:')
    
    axis_widget = widgets.Dropdown(options=['x', 'y', 'z'], value='z', description='Slice Axis:')
    label_type_widget = widgets.Dropdown(
        options=[('Labels', LabelType.LABELS), ('Supervoxels', LabelType.SUPERVOXELS)],
        value=LabelType.LABELS,
        description='Label Type:'
    )
    
    def visualize_slice(x1, y1, z1, x2, y2, z2, axis, label_type):
        """Visualize a 2D slice with the given parameters."""
        if x2 <= x1 or y2 <= y1 or z2 <= z1:
            print("Error: Bottom-right coordinates must be greater than top-left coordinates")
            return
        
        print(f"Extracting {axis}-axis slice from ({x1},{y1},{z1}) to ({x2},{y2},{z2})...")
        
        try:
            slice_data = extract_2d_slice(reader, (x1, y1, z1), (x2, y2, z2), axis, label_type=label_type)
            
            # Create visualization
            plt.figure(figsize=(12, 8))
            
            if slice_data.max() > 0:  # If there's actual data
                cmap, unique_labels = create_pseudocolor_map(slice_data)
                
                plt.imshow(slice_data, cmap=cmap, interpolation='nearest')
                plt.colorbar(label='Label ID')
                
                print(f"Labels found: {len(unique_labels)} unique labels")
                print(f"Label range: {unique_labels.min()} to {unique_labels.max()}")
            else:
                plt.imshow(slice_data, cmap='gray', interpolation='nearest')
                plt.colorbar(label='Label ID')
                print("No labels found in this slice (all zeros)")
            
            # Set axis labels based on slice direction
            if axis == 'x':
                plt.xlabel('Y coordinate')
                plt.ylabel('Z coordinate')
                plt.title(f'X-axis slice (shape: {slice_data.shape})')
            elif axis == 'y':
                plt.xlabel('X coordinate')
                plt.ylabel('Z coordinate')
                plt.title(f'Y-axis slice (shape: {slice_data.shape})')
            else:
                plt.xlabel('X coordinate')
                plt.ylabel('Y coordinate')
                plt.title(f'Z-axis slice (shape: {slice_data.shape})')
            
            plt.tight_layout()
            plt.show()
            
        except Exception as e:
            print(f"Error creating slice: {e}")
    
    # Create interactive widget
    interactive_widget = widgets.interact(
        visualize_slice,
        x1=x1_widget, y1=y1_widget, z1=z1_widget,
        x2=x2_widget, y2=y2_widget, z2=z2_widget,
        axis=axis_widget, label_type=label_type_widget
    )
    
else:
    print("Cannot create visualization - reader not initialized")

## Manual Testing

For more precise control, you can manually specify coordinates:

In [ ]:
# Manual slice extraction example
if reader is not None:
    # Define your region of interest
    TOP_LEFT = (0, 0, 0)      # Replace with your coordinates
    BOTTOM_RIGHT = (256, 256, 256)  # Replace with your coordinates
    SLICE_AXIS = 'z'          # 'x', 'y', or 'z'
    SLICE_POSITION = 128      # Position along the slice axis
    
    print(f"Manual slice extraction:")
    print(f"  Region: {TOP_LEFT} to {BOTTOM_RIGHT}")
    print(f"  Axis: {SLICE_AXIS}, Position: {SLICE_POSITION}")
    
    try:
        # Extract slice
        slice_data = extract_2d_slice(
            reader, TOP_LEFT, BOTTOM_RIGHT, 
            axis=SLICE_AXIS, slice_pos=SLICE_POSITION,
            label_type=LabelType.LABELS
        )
        
        # Visualize
        plt.figure(figsize=(10, 8))
        
        if slice_data.max() > 0:
            cmap, unique_labels = create_pseudocolor_map(slice_data)
            im = plt.imshow(slice_data, cmap=cmap, interpolation='nearest')
            plt.colorbar(im, label='Label ID')
            
            print(f"\nResults:")
            print(f"  Slice shape: {slice_data.shape}")
            print(f"  Labels found: {len(unique_labels)}")
            print(f"  Label range: {unique_labels.min()} to {unique_labels.max()}")
            
            # Show some statistics
            non_zero = slice_data[slice_data > 0]
            if len(non_zero) > 0:
                print(f"  Non-zero pixels: {len(non_zero)} ({100*len(non_zero)/slice_data.size:.1f}%)")
        else:
            plt.imshow(slice_data, cmap='gray', interpolation='nearest')
            plt.colorbar(label='Label ID')
            print(f"\nSlice shape: {slice_data.shape} - No labels found (all zeros)")
        
        plt.title(f'{SLICE_AXIS.upper()}-axis slice at position {SLICE_POSITION}')
        plt.xlabel('X coordinate' if SLICE_AXIS != 'x' else 'Y coordinate')
        plt.ylabel('Y coordinate' if SLICE_AXIS == 'z' else 'Z coordinate')
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"Error: {e}")
else:
    print("Reader not available for manual testing")

## Chunk Information

Explore individual chunks and their properties:

In [ ]:
if reader is not None and reader.available_chunks:
    # Examine a specific chunk
    chunk_x, chunk_y, chunk_z = reader.available_chunks[0]  # First available chunk
    
    print(f"Examining chunk ({chunk_x}, {chunk_y}, {chunk_z}):")
    
    try:
        # Get chunk info without decompressing
        chunk_info = reader.get_chunk_info(chunk_x, chunk_y, chunk_z)
        print(f"  Labels count: {chunk_info['labels_count']}")
        print(f"  Compressed size: {chunk_info['compressed_size']} bytes")
        
        # Read raw data
        raw_data = reader.read_chunk_raw(chunk_x, chunk_y, chunk_z)
        print(f"  Raw labels: {len(raw_data['labels'])} unique labels")
        print(f"  Raw supervoxels: {len(raw_data['supervoxels'])} unique supervoxels")
        
        # Read decompressed data
        labels_data = reader.read_chunk(chunk_x, chunk_y, chunk_z, label_type=LabelType.LABELS)
        supervoxel_data = reader.read_chunk(chunk_x, chunk_y, chunk_z, label_type=LabelType.SUPERVOXELS)
        
        print(f"  Decompressed shape: {labels_data.shape}")
        print(f"  Labels data type: {labels_data.dtype}")
        print(f"  Unique labels in chunk: {len(np.unique(labels_data))}")
        print(f"  Unique supervoxels in chunk: {len(np.unique(supervoxel_data))}")
        
        # Show a small slice from the chunk
        mid_z = labels_data.shape[2] // 2
        chunk_slice = labels_data[:, :, mid_z]
        
        if chunk_slice.max() > 0:
            plt.figure(figsize=(8, 6))
            cmap, unique_labels = create_pseudocolor_map(chunk_slice)
            plt.imshow(chunk_slice, cmap=cmap, interpolation='nearest')
            plt.colorbar(label='Label ID')
            plt.title(f'Middle Z-slice of chunk ({chunk_x}, {chunk_y}, {chunk_z})')
            plt.xlabel('X coordinate')
            plt.ylabel('Y coordinate')
            plt.show()
        else:
            print(f"  No labels in middle Z-slice of this chunk")
        
    except Exception as e:
        print(f"  Error reading chunk: {e}")
        
else:
    print("No chunks available for examination")

## Performance Testing

Test the performance of reading multiple chunks:

In [ ]:
import time

if reader is not None and len(reader.available_chunks) > 5:
    print("Performance test - reading first 5 chunks:")
    
    total_size = 0
    total_labels = 0
    
    start_time = time.time()
    
    for i, (chunk_x, chunk_y, chunk_z) in enumerate(reader.available_chunks[:5]):
        chunk_start = time.time()
        
        try:
            # Read chunk
            chunk_data = reader.read_chunk(chunk_x, chunk_y, chunk_z, label_type=LabelType.LABELS)
            
            chunk_time = time.time() - chunk_start
            chunk_size = chunk_data.nbytes
            unique_labels = len(np.unique(chunk_data))
            
            total_size += chunk_size
            total_labels += unique_labels
            
            print(f"  Chunk {i+1} ({chunk_x}, {chunk_y}, {chunk_z}): "
                  f"{chunk_time:.3f}s, {chunk_size/1024/1024:.1f}MB, {unique_labels} labels")
            
        except Exception as e:
            print(f"  Chunk {i+1} ({chunk_x}, {chunk_y}, {chunk_z}): Error - {e}")
    
    total_time = time.time() - start_time
    
    print(f"\nSummary:")
    print(f"  Total time: {total_time:.3f}s")
    print(f"  Total data: {total_size/1024/1024:.1f}MB")
    print(f"  Average throughput: {total_size/1024/1024/total_time:.1f}MB/s")
    print(f"  Total unique labels: {total_labels}")
    
else:
    print("Not enough chunks available for performance testing")